# 02 - Refusal direction on Gemma-2-2b-it (Colab / Kaggle T4)

Same pipeline as `01_qwen_local.ipynb`, run on a free T4 because Gemma-2 overflows in fp16 and
does not fit a 6 GB card in fp32.

**Before running (Colab):**

1. *Runtime > Change runtime type > T4 GPU.*
2. Accept the Gemma licence on Hugging Face (https://huggingface.co/google/gemma-2-2b-it) with the account whose token you use.
3. Add that token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar) and enable notebook access. On Kaggle, add it as a secret with the same name, or paste it when `huggingface_hub.login()` prompts.
4. Run all cells. Expect roughly 30-60 minutes for the full 26-layer sweep (see the dtype note below).

**After running:** download the zip produced by the last cell, unpack it into `results/gemma-2-2b-it/` in the repo, and commit it; then update the README results table from `summary.json` and `refusal_by_layer.csv`.

Set `REFUSAL_SMOKE=1` in the environment to run this notebook on CPU with `EleutherAI/pythia-70m`
and tiny splits; that is how the cells were tested without a GPU.

In [ ]:
import os, sys, subprocess, json
SMOKE = os.environ.get("REFUSAL_SMOKE") == "1"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("colab:", IN_COLAB, "| smoke mode:", SMOKE)

In [ ]:
# Clone the repo (for data/splits_seed0.json and results/) and install the package.
# Colab ships its own torch / numpy / pandas / matplotlib; those are kept rather than
# reinstalled to the versions pinned in pyproject.toml. The versions actually used are
# written to results/<model>/summary.json.
REPO_URL = "https://github.com/ORmanius/refusal-direction-replication.git"
if IN_COLAB:
    if not os.path.isdir("refusal-direction-replication"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir("refusal-direction-replication")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nnsight==0.8.2", "transformers==5.22.1", "accelerate==1.15.0"], check=True)
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (REPO / "pyproject.toml").exists(), f"expected the repo root, got {REPO}"
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
print("repo:", REPO)

In [ ]:
# Hugging Face token for the gated Gemma weights.
import re

def clean_token(raw):
    """Strip whitespace, quotes and stray punctuation a paste can pick up; None if nothing is left."""
    if raw is None:
        return None
    tok = raw.strip().strip("\"'").strip(" ,;\t\r\n")
    return tok or None

raw_env = os.environ.get("HF_TOKEN")
HF_TOKEN = clean_token(raw_env)
if raw_env and not HF_TOKEN:
    print("HF_TOKEN in the environment holds no token (only punctuation or whitespace); ignoring it.")
os.environ.pop("HF_TOKEN", None)  # never let a garbled value reach huggingface_hub
if IN_COLAB and not HF_TOKEN:
    from google.colab import userdata
    raw_secret = userdata.get("HF_TOKEN")
    HF_TOKEN = clean_token(raw_secret)
    if raw_secret and not HF_TOKEN:
        print("The Colab secret HF_TOKEN holds no token (only punctuation or whitespace). "
              "Open the key icon in the left sidebar and paste the hf_... value again.")
if HF_TOKEN and not re.fullmatch(r"hf_[A-Za-z0-9]+", HF_TOKEN):
    raise ValueError(
        "HF_TOKEN does not look like a Hugging Face token (expected hf_... with letters and digits only). "
        "Re-copy the token from https://huggingface.co/settings/tokens into the Colab secret, "
        "with no spaces, commas or line breaks."
    )
if not SMOKE and not HF_TOKEN:
    from huggingface_hub import login
    login()  # interactive prompt
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("token set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
import torch
from refusal_direction import __version__
from refusal_direction.data import get_splits
from refusal_direction.model import load_model, format_prompts
from refusal_direction.pipeline import ExperimentConfig, run_experiment, config_summary
from refusal_direction.scoring import is_refusal

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), "| bf16 supported (incl. emulation):", torch.cuda.is_bf16_supported())
else:
    print("no GPU")
print("refusal_direction", __version__, "| torch", torch.__version__)

## dtype on a T4

A T4 is Turing (sm_75) like the RTX 2060: it has **no native bf16**. Recent PyTorch reports bf16 as
supported on it through emulation, which works but is slow and not what Gemma was trained on;
fp16 overflows in Gemma-2. The safe choice is **float32**: the 2.6B weights take about 10.5 GB of the
T4's 15 GB, leaving room for batches of 8 prompts. Switch `DTYPE` to `"bfloat16"` if you want to try
the faster path; the results files record which was used.

In [ ]:
DTYPE = "float32"
if SMOKE:
    MODEL, DTYPE, N_TRAIN, N_VAL, LAYERS, BATCH = "EleutherAI/pythia-70m", "float32", 8, 4, [0, 1], 4
    OUT = REPO / "results" / "_smoke_pythia-70m"
    SPLITS = REPO / "results" / "_smoke_pythia-70m" / "splits_smoke.json"
    ATTN = None
else:
    MODEL, N_TRAIN, N_VAL, LAYERS, BATCH = "google/gemma-2-2b-it", 128, 32, None, 8
    OUT = REPO / "results" / "gemma-2-2b-it"
    SPLITS = REPO / "data" / "splits_seed0.json"
    ATTN = "eager"  # recommended for Gemma-2 (attention logit soft-capping)

cfg = ExperimentConfig(
    model_name=MODEL, out_dir=OUT, dtype=DTYPE, n_train=N_TRAIN, n_val=N_VAL, seed=0,
    position=-1, max_new_tokens=64, batch_size=BATCH, layers=LAYERS,
    attn_implementation=ATTN, splits_path=SPLITS, hf_token=os.environ.get("HF_TOKEN"),
)
splits = get_splits(cfg.splits_path, cfg.n_train, cfg.n_val, cfg.seed)
print({k: len(v) for k, v in splits.items() if k.startswith("harm")})
print(json.dumps(config_summary(cfg), indent=1))  # paths shown relative to the repo

In [ ]:
bundle = load_model(cfg.model_name, dtype=getattr(torch, cfg.dtype),
                    attn_implementation=cfg.attn_implementation, hf_token=cfg.hf_token)
print(f"{bundle.name}: {bundle.n_layers} layers, d_model={bundle.d_model}, {bundle.dtype}, {bundle.device}")
if torch.cuda.is_available():
    print("GPU memory allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1e9))
print(repr(format_prompts(bundle, [splits["harmless_val"][0]])[0]))

In [ ]:
res = run_experiment(cfg, bundle=bundle)
summary = res["summary"]
print(json.dumps({k: v for k, v in summary.items() if k != "config"}, indent=1))

In [ ]:
df = res["df"]
df.style.format({c: "{:.3f}" for c in df.columns if "rate" in c or c == "direction_norm"}).hide(axis="index")

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(cfg.out_dir / "refusal_by_layer.png")))

In [ ]:
def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8")]

add = load_jsonl(cfg.out_dir / "completions_harmless_addition.jsonl")
base = load_jsonl(cfg.out_dir / "completions_harmless_baseline.jsonl")
for b, a in list(zip(base, add))[:4]:
    print("PROMPT:   ", a["instruction"])
    print("baseline: ", b["completion"][:160].replace("\n", " "), "| refusal =", is_refusal(b["completion"]))
    print("+direction:", a["completion"][:160].replace("\n", " "), "| refusal =", is_refusal(a["completion"]))
    print()
abl = load_jsonl(cfg.out_dir / "completions_harmful_ablation.jsonl")
print("per-prompt refusal flags on harmful val after ablation:",
      "".join("R" if is_refusal(x["completion"]) else "." for x in abl))

In [ ]:
# Package the results for the repo (completions_harmful_ablation.jsonl is left out on purpose).
import shutil, zipfile
zip_path = REPO / f"{cfg.out_dir.name}_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(cfg.out_dir.iterdir()):
        if p.name not in ("completions_harmful_ablation.jsonl", "directions.pt"):
            z.write(p, arcname=f"{cfg.out_dir.name}/{p.name}")
print("wrote", zip_path, zip_path.stat().st_size // 1024, "KB")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))